In [25]:
# Extract_Arrlington_Soldiers.py
# 5/13/2025

from bs4 import BeautifulSoup 
import datetime
from datetime import datetime as dt
from datetime import date 
from datetime import timedelta
import time
import re
from pandas import read_csv
import pandas as pd
# from selenium import webdriver
from seleniumwire import webdriver
from selenium.webdriver.common.by import By
import json
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
import sys
import random
from zenrows import ZenRowsClient
import requests
import math
import numpy as np
import urllib3
import inspect
# from IPython.display import display


# def maximize_screen():
#     from IPython.display import display, HTML
#     display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def left(s, amount):
    return s[:amount]
def right(s, amount):
    return s[-amount:]
def mid(s, offset, amount):
    return s[offset:offset+amount]

def maximize_screen():
    from IPython.display import display, HTML
    display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def timestamp():
    today=str(datetime.date.today()).split('-')
    year=today[0]
    month=today[1]
    day=today[2]
    a=str(datetime.datetime.now())
    time_split=re.split(':| ',a)
    hour=time_split[1]
    timestamp=year+month+day+hour
    return timestamp

def convert_date_z(date): 
    date_t= pd.to_datetime(date, utc=True)
    date_out=date_t.dt.strftime("%m/%d/%Y")
    return date_out

def convert_datetime_t(event_date_t):
    no_time='2000-01-01T00:00:00'
    no_time_dt=dt.strptime(no_time, "%Y-%m-%dT%H:%M:%S") 
    event_date_t = event_date_t.fillna(no_time)
    event_date_dt=pd.to_datetime(event_date_t, format="%Y-%m-%dT%H:%M:%S")
    event_date_index=event_date_dt.dt.strftime("%Y%m%d")
    event_date =event_date_dt.dt.strftime("%m/%d/%Y")
    event_date_rnd_dt = event_date_dt + timedelta(minutes=30)
    event_time_index=event_date_rnd_dt.dt.strftime('%H')  
    event_time = event_date_dt.dt.strftime('%H:%M')
    return     event_date,event_date_index,event_time,event_time_index 



def random_sleep(call_count): 
    global sleep_time0,listings_df
    if call_count%10==0:
        sleep_time =sleep_time0
        now_date = datetime.datetime.now() + timedelta(seconds=sleep_time)
        sleep_end_date=now_date.strftime("%m/%d/%Y %H:%M:%S")    
        print(f"\t\t\tSleeping for {sleep_time:.0f} seconds • Sleeping till: {sleep_end_date}")
    else:
        sleep_time = random.uniform(15, 30)
    return sleep_time 

def get_total_cemeteries(url):
    global my_header
    response = requests.request("GET", url, headers=my_header) 
    status_code=response.status_code
    url_source = response.text 
    
    beautifulSoupText = BeautifulSoup(url_source, 'html.parser') 
    specific_tag_text = beautifulSoupText.get_text()
    capture_json=json.loads(specific_tag_text) 
    total_cemeteries=capture_json['data']['cemeteries']['total']
    return total_cemeteries

def get_grave_count(url):
    global status_code
    capture_json=[]
    grave_count=0
    response = requests.request("GET", url, headers=my_header) 
    status_code=response.status_code
    url_source = response.text 
    if status_code!=200:print('\tstatus_code:',status_code, 'url',url);return capture_json,grave_count
    beautifulSoupText = BeautifulSoup(url_source, 'html.parser') 
    specific_tag_text = beautifulSoupText.get_text()
    capture_json=json.loads(specific_tag_text) 
    grave_count=capture_json['SearchResult']['TotalCount']
    return capture_json,grave_count
    
def extract_grave_data(capture_json):
    global field_list_set,field_list
    collection_df=pd.json_normalize(capture_json['SearchResult']['Records'])
    collection_columns_set=set(list(collection_df))
    missing_columns=field_list_set-collection_columns_set
    collection_df[list(missing_columns)]=np.nan
    grave_frame_df=collection_df[field_list]
    return grave_frame_df

def clean_dataframe(graves_df):
    global field_list
    for column in field_list:
        try:
            graves_df[column]= graves_df[column].astype(str).replace(r"\,", ' * ', regex=True).str.strip()
            graves_df[column]= graves_df[column].astype(str).replace(r"\'", '', regex=True).str.strip()
        except:
            continue
    delete_list=[r'\"',r"\'",r'\?',r'\[',r'\]']
    for item in delete_list:
        for column in field_list:
            try:
                graves_df[column]= graves_df[column].astype(str).replace(item, ' ', regex=True).str.strip()
            except:
                continue
    return graves_df

#*************** Append  *******************************
append=0
# append_flag=input('To Append Press 1')
# if append_flag=='1':append=1
#*************** Append  *******************************

my_header = {
  'content-type': 'application/json',
  'priority': 'u=1, i',
  'sec-ch-ua': '"Google Chrome";v="135", "Not-A.Brand";v="8", "Chromium";v="135"',
  'sec-ch-ua-platform': '"Windows"',
  'sec-fetch-mode': 'cors',
  'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36',
  'Accept': 'application/json, text/plain, */*',
  'Accept-language': 'en-GB,en-US;q=0.9,en;q=0.8,ja;q=0.7',
  'referer': 'https://ancexplorer.army.mil/publicwmv/index.html'
}
maximize_screen()
timestamp=timestamp()
collection_date0=datetime.datetime.now().strftime("%m/%d/%Y")
collection_date_dt=pd.to_datetime(collection_date0)
collection_date=collection_date0

start_time=datetime.datetime.now()
event_time=start_time

process_name='Extract Arrlington_Soldiers '
print('**Starting *** ',process_name,' *** , Time:',start_time,' **\n')
DirMain="C:\\Users\\jrozm\\OneDrive\\Benjamin\\"



# fileIn="JewishCemeteries_CountyCorrelated"
fileOutput="Arrlington_Soldiers"

# if append==1:fileIn='JewishCemeteries_CountyCorrelated_X'


# fileCemeteriesIn=DirMain+"OutputData\\Cemeteries\\"+fileIn+".csv"
fileOut=DirMain+"OutputData\\ABMC\\"+fileOutput+"_"+timestamp+".csv"

fileOut0=DirMain+"OutputData\\ABMC\\"+fileOutput+".csv"
fileOutInt=DirMain+"OutputData\\ABMC\\"+fileOutput+"_Interim.csv"


field_list=['PRIMARYLASTNAME','PRIMARYFIRSTNAME','PRIMARYMIDDLENAME','BRANCHOFSERVICE','DECEDENTINTERMENTID','ISS_ID','CemeteryId','CemeteryName',\
  'DOB','DOD','DOI','COLUMN','COURT','SECTION','CEMETERYSECTION','COLUMBARIUMSECTION','GRAVE','GRAVEROW','LOCATIONID','NICHE','SUFFIX',\
 'RELATIONSHIP', 'Rank','IMAGEURL','Images']
field_list_set=set(field_list)
if append==1:print('**** Appending ****')    
if append!=1:graves_df=pd.DataFrame([])  

sleep_time0=3*60
graves_df=pd.DataFrame([])
get_counter=0
frame_count=0


if append==1:grave_frame_count=25001
grave_count=500
max_records=100
url0="https://ancexplorer.army.mil/proxy/proxy.ashx?https://wspublic.eiss.army.mil/v1/IssRetrieveServices.svc/search?AppId=Roi&q=dod%3D_%3A_%3A"
url1 ="%2Ccemeteryid%3D46&start="
url2 ="&limit=1"+str(max_records)+"&sortColumn=PrimaryLastName%2CPrimaryFirstName%2CPrimaryMiddleName%2CDOB%2CDOD&sortOrder=asc&f=json"

search_years_list=['1917','1918','1919','1940','1941','1942','1943','1944','1945','1946',\
              '1950','1951','1952','1953','1964','1965','1966','1967','1968','1969','1970','1971','1972']
for search_year in search_years_list:
    grave_frame_count=0
    while grave_frame_count<grave_count:
        # print('Point 1 : search_year,grave_frame_count,grave_count:',search_year,grave_frame_count,grave_count)
        url=url0+search_year+url1+str(grave_frame_count)+url2
        capture_json,grave_count=get_grave_count(url)       
        # print(url)
        # print(json.dumps(capture_json))            
        # input('capture_json')         
        if status_code!=200:print('\tstatus_code:',status_code);break 
   
        grave_frame_df=extract_grave_data(capture_json)
        grave_frame_df['DOD'] = pd.to_datetime(grave_frame_df['DOD'], format="%m/%d/%Y %H:%M", errors='coerce')
        grave_frame_df['DOD']=grave_frame_df['DOD'].dt.strftime("%m/%d/%Y")
        graves_df = graves_df._append(grave_frame_df, ignore_index = True) 
        
        grave_frame_count+=len(grave_frame_df)
        # print('Point 2 : search_year,grave_frame_count,grave_count:',search_year,grave_frame_count,grave_count)
        # input('')
        
        graves_df=clean_dataframe(graves_df)
        graves_df.to_csv(fileOutInt,mode="w", index=False,header=True)
        print(f"    --- {process_name:<20}• Search Year: {search_year} • grave count: {grave_frame_count}/{grave_count} ")
        get_counter+=1
        sleep_time=random_sleep(get_counter)       
        time.sleep(sleep_time) 
    print(f" **** {process_name:<20}• Search Year: {search_year}  • grave count: {grave_count} ")
end_time=datetime.datetime.now()
processing_time=end_time-start_time
processing_time_seconds=int(processing_time.total_seconds())
print(f"**** {process_name}   Processing Time: {processing_time_seconds} Sec")   
graves_df=clean_dataframe(graves_df)  
graves_df.to_csv(fileOut,mode="w", index=False,header=True)
graves_df.to_csv(fileOut0,mode="w", index=False,header=True) 
print("\n")
print('Uploaded To: ',fileOut0)



**Starting ***  Extract Arrlington_Soldiers   *** , Time: 2025-05-13 15:41:23.894698  **

    --- Extract Arrlington_Soldiers • Search Year: 1917 • grave count: 521/521 
 **** Extract Arrlington_Soldiers • Search Year: 1917  • grave count: 521 
    --- Extract Arrlington_Soldiers • Search Year: 1918 • grave count: 1100/5346 
    --- Extract Arrlington_Soldiers • Search Year: 1918 • grave count: 2200/5346 
    --- Extract Arrlington_Soldiers • Search Year: 1918 • grave count: 3300/5346 
    --- Extract Arrlington_Soldiers • Search Year: 1918 • grave count: 4400/5346 
    --- Extract Arrlington_Soldiers • Search Year: 1918 • grave count: 5346/5346 
 **** Extract Arrlington_Soldiers • Search Year: 1918  • grave count: 5346 
    --- Extract Arrlington_Soldiers • Search Year: 1919 • grave count: 1067/1067 
 **** Extract Arrlington_Soldiers • Search Year: 1919  • grave count: 1067 
    --- Extract Arrlington_Soldiers • Search Year: 1940 • grave count: 1100/1250 
    --- Extract Arrlington_So

In [24]:
grave_frame_df

,PRIMARYLASTNAME,PRIMARYFIRSTNAME,PRIMARYMIDDLENAME,BRANCHOFSERVICE,DECEDENTINTERMENTID,ISS_ID,CemeteryId,CemeteryName,DOB,DOD,...,COLUMBARIUMSECTION,GRAVE,GRAVEROW,LOCATIONID,NICHE,SUFFIX,RELATIONSHIP,Rank,IMAGEURL,Images
0,ALDEN,HELEN,P,,287669,287753,46,Arlington National Cemetery,,02/24/1917,...,,18926,,17:_:18926:_:_:_,,,Data Under Review,,Headstone Front_411618.jpg|https://wspublic.ei...,"[{'RemainsLocationImageId': 411618, 'ImageType..."
1,ALDEN,WILLIAM,H,,287668,287752,46,Arlington National Cemetery,,02/24/1917,...,,18926,,17:_:18926:_:_:_,,,Data Under Review,,Headstone Front_411610.jpg|https://wspublic.ei...,"[{'RemainsLocationImageId': 411610, 'ImageType..."
2,ALLEN,DORA,Z,,255364,255443,46,Arlington National Cemetery,,04/26/1917,...,,206,,1:_:206:_:_:_,,,Spouse,,Headstone Front_1306151.jpg|https://wspublic.e...,"[{'RemainsLocationImageId': 1306151, 'ImageTyp..."
3,ALLEN,EDWARD,G,USN,260969,261050,46,Arlington National Cemetery,12/04/1859 00:00,09/11/1917,...,,1530-ES,,3:_:1530-ES:_:_:_,,,Veteran (Self),ASST ENGR,Headstone Front_954780.jpg|https://wspublic.ei...,"[{'RemainsLocationImageId': 954780, 'ImageType..."
4,ANDERSON,HARRY,L,USN,93836,93884,46,Arlington National Cemetery,,08/25/1917,...,,21804,,23:_:21804:_:_:_,,,Data Under Review,CAB STD,Headstone Front_521967.jpg|https://wspublic.ei...,"[{'RemainsLocationImageId': 521967, 'ImageType..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
516,WRIGHT,ALLEN,,CAV,285841,285925,46,Arlington National Cemetery,,06/21/1917,...,,18999,,17:_:18999:_:_:_,,,Veteran (Self),Other,Headstone Front_832111.jpg|https://wspublic.ei...,"[{'RemainsLocationImageId': 832111, 'ImageType..."
517,WRIGHT,CARRIE,F,,285963,286047,46,Arlington National Cemetery,,09/06/1917,...,,17079,,17:_:17079:_:_:_,,,Data Under Review,,Headstone Front_654751.jpg|https://wspublic.ei...,"[{'RemainsLocationImageId': 654751, 'ImageType..."
518,WRIGHT,WILLIAM,H,,285792,285876,46,Arlington National Cemetery,,10/06/1917,...,,19036,,17:_:19036:_:_:_,,,Data Under Review,,Headstone Front_371520.jpg|https://wspublic.ei...,"[{'RemainsLocationImageId': 371520, 'ImageType..."
519,YOUNG,CARRINGTON,A,,260793,260874,46,Arlington National Cemetery,,11/11/1917,...,,2458,,3:_:2458:_:_:_,,,Data Under Review,,Headstone Front_573182.jpg|https://wspublic.ei...,"[{'RemainsLocationImageId': 573182, 'ImageType..."


In [ ]:
collection_df=pd.DataFrame(capture_json['csw:GetRecordsResponse']['csw:SearchResults']['iStoreRecord'])
collection_columns_set=set(list(collection_df))
missing_columns=field_list_set-collection_columns_set
collection_df[list(missing_columns)]=np.nan
grave_frame_df=collection_df[field_list]